In [ ]:
# --- ITERATION BANNER (update this string each push) ---
ITERATION = "fix trajectory-log unpack (history is now 5-tuples)"
print(f"\n{'═' * 72}\n  running: {ITERATION}\n{'═' * 72}\n")

# Demo on Colab — mirrors the production mission flow

This notebook walks through the exact same sequence the real robot will run, just with photo uploads standing in for hardware.

**Production flow:**

| Phase | What happens in production | What you do here |
|---|---|---|
| 1. Report | A reporter snaps a tight crop of the trash + a wider context shot, uploads via Expo app to the relay. | Upload your own REF + CTX photos (or use the bundled test pair). |
| 2. Navigate | Robot's mounted phone drives the robot via Apple Maps waypoints + GPS to the report's location. | Skipped — A100 has no GPS / motors. Assume we've arrived. |
| 3. Approach | At the last waypoint, the brain takes over the WebSocket to the Pi. Each camera frame → OWLv2 (find target) → if hidden, Qwen3-VL scout (which way to rotate) → discrete Action → motor PWM. | Upload live-style frames one at a time. The brain processes each, shows the Action. Controller state (search bursts, etc.) carries across frames. |
| 4. Verify / report | Robot confirms target gone (driven over with scoop). POST result to relay. | Trajectory cell shows whether STOP was reached, mirroring the success signal. |

**Setup:** Runtime → Change runtime type → **A100 GPU**, then run cells top to bottom.

## 0. Set up the Colab runtime (once per session)

Clones the repo + installs HuggingFace transformers. ~2 min.

In [ ]:
!git clone https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git 2>/dev/null || echo 'repo already cloned'
%cd RU-IEEE-Hackathon

# Upgrade transformers + helpers, but do NOT -U Pillow — Colab preinstalls gradio
# which pins pillow<12, and a floating -U pulls in 12.x and breaks `from PIL import ImageText`.
!pip install -q git+https://github.com/huggingface/transformers@main accelerate bitsandbytes
# Pin Pillow to a known-good version (undoes any partial upgrade from a previous cell).
!pip install -q --force-reinstall --no-deps Pillow==10.4.0
print('\ndone. Proceed to the next cell.')
print('If a later cell throws a PIL/Pillow ImportError: Runtime → Restart session, then re-run from cell 1 onward.')

## 1. Load the brain (once per session)

Loads OWLv2 + Qwen3-VL-8B at fp16. ~3-5 min. After this, every step is fast.

**Production analog:** This is the brain process starting up on the RTX 4080. In production it boots once at the start of the day and stays loaded for hours.

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

REPO = Path.cwd()
sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'no GPU — Runtime → Change runtime type → A100'
print(f'GPU: {torch.cuda.get_device_name(0)}  ·  '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')

from brain.perception.target_finder import TargetFinder
from brain.perception.vlm_scout import VLMScout
from brain.control.loop import Action, ApproachController
from brain.control.action_to_pwm import pwm_for

print('loading OWLv2 (~300 MB)...')
finder = TargetFinder(model_name='google/owlv2-base-patch16-ensemble', min_sim=0.3)

print('loading Qwen3-VL-8B at fp16 (~18 GB on GPU)...')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=False)
print(f'  loaded in {time.monotonic() - t0:.1f}s  ·  '
      f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('\nbrain ready. No mission loaded yet — go to the next cell.')

## 2. Load the mission's reference photos

**Production analog:** the reporter submits two photos (tight crop + wider scene) through the Expo app. In this notebook they live in the `references/` folder of the repo — whatever you committed last will be used.

Currently using `references/ref.jpg` (tight crop) and `references/ctx.jpg` (wider scene). If you want to swap in new ones, replace those files on your local machine, commit + push, then re-run the clone cell.

In [ ]:
REF = REPO / 'references' / 'ref.jpg'
CTX = REPO / 'references' / 'ctx.jpg'
assert REF.exists() and CTX.exists(), 'missing reference files in references/'

ref_bgr = cv2.imread(str(REF))
ctx_bgr = cv2.imread(str(CTX))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'REF (target)\n{REF.name}')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(ctx_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'CTX (wider scene)\n{CTX.name}')
axes[1].axis('off')
plt.show()

## 3. Mission start — bind references into the brain

**Production analog:** When the brain accepts a new job, it loads the reporter's REF into OWLv2 (encodes once, caches the query embedding) and creates a fresh ApproachController bound to both REF and CTX. This cell does both.

In [ ]:
finder.load_reference(ref_bgr)
controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)
history = []
print('mission armed. Brain is now hunting for THIS specific target.')
print('Next: feed it live frames.')

## 4. Approach loop — feed live frames, get Actions

**Production analog:** The brain reads frames from the Pi's MJPEG stream at ~15 FPS. For each frame, `controller.step(frame)` returns one Action. That action gets translated to PWM via `action_to_pwm.py` and sent to the Pi over WebSocket. The Pi's L298N drives the motors.

Here, you upload "live" frames one at a time. The controller's internal state — including the 15-frame SEARCH burst counter that prevents calling Qwen3-VL on every frame — **persists across uploads**, exactly as it would across real camera frames.

**Tip:** take 5-10 phone photos of the bottle scene at varying poses BEFORE starting:
- bottle hidden / out of frame
- bottle at the edge of frame
- bottle centered far away
- bottle centered close up
- bottle filling the frame (close enough to STOP)

Upload them sequentially — that's the closest analog to a real camera stream.

In [ ]:
MIN_SIM = 0.3   # raise to 0.6+ if OWLv2 saturates with false positives
finder.min_sim = MIN_SIM

ACTION_LABEL = {
    Action.FORWARD:      'WALK FORWARD',
    Action.LEFT:         'TURN LEFT',
    Action.RIGHT:        'TURN RIGHT',
    Action.STOP:         'STOP \u2014 TARGET REACHED',
    Action.SEARCH_LEFT:  'ROTATE LEFT (scanning)',
    Action.SEARCH_RIGHT: 'ROTATE RIGHT (scanning)',
}
ACTION_COLOR = {
    Action.FORWARD: '#00cc00', Action.LEFT: '#ff8800',
    Action.RIGHT: '#ff8800', Action.STOP: '#cc0000',
    Action.SEARCH_LEFT: '#cc00cc', Action.SEARCH_RIGHT: '#cc00cc',
}
ACTION_ARROW = {
    Action.FORWARD: '\u2191', Action.LEFT: '\u2190',
    Action.RIGHT: '\u2192', Action.STOP: '\u25A0',
    Action.SEARCH_LEFT: '\u21BA', Action.SEARCH_RIGHT: '\u21BB',
}

def process_frame(frame, source_label=''):
    """Run the brain on one BGR frame, display the Action."""
    t0 = time.monotonic()
    detections = finder.detect(frame)
    action = controller.step(frame)
    elapsed_ms = (time.monotonic() - t0) * 1000
    left, right = pwm_for(action)
    tops = [(float(d.confidence), tuple(int(v) for v in d.xyxy)) for d in detections[:3]]
    history.append((action, len(detections), elapsed_ms, source_label, tops))

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.05)
    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if detections:
        d = detections[0]
        x1, y1, x2, y2 = d.xyxy
        ax_img.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor='lime', linewidth=4))
        ax_img.text(x1, max(y1-8, 16), f'target  {d.confidence:.2f}',
                    color='black', fontsize=14, weight='bold',
                    bbox=dict(facecolor='lime', alpha=0.85, pad=4))
    ax_img.set_title(source_label, fontsize=11, color='#666', loc='left')
    ax_img.axis('off')

    ax_act = fig.add_subplot(gs[1])
    ax_act.set_xlim(0, 1); ax_act.set_ylim(0, 1); ax_act.axis('off')
    color = ACTION_COLOR[action]
    ax_act.text(0.5, 0.65,
                f'{ACTION_ARROW[action]}  {ACTION_LABEL[action]}  {ACTION_ARROW[action]}',
                ha='center', va='center', fontsize=44, color=color, weight='bold')
    ax_act.text(0.5, 0.2,
                f'frame #{len(history)}   \u00b7   {len(detections)} detection(s)   \u00b7   '
                f'pwm = ({left:+d}, {right:+d})   \u00b7   {elapsed_ms:.0f} ms',
                ha='center', va='center', fontsize=13, color='#666')
    plt.show()
    return action

# Use the bundled live.jpg by default — no upload needed.
live_path = REPO / 'references' / 'live.jpg'
assert live_path.exists(), 'missing references/live.jpg'
live_frame = cv2.imread(str(live_path))
process_frame(live_frame, source_label=str(live_path.name))

**Re-run the cell above** to process `references/live.jpg` again (useful after tweaking `MIN_SIM`). The controller's internal state carries over between cell runs.

Or drop `references/live_001.jpg`, `live_002.jpg`, ... into the repo for an ordered sequence, then run the cell below.

In [ ]:
# Optional: process ANY additional frames dropped into references/live_*.jpg
# (e.g. live_001.jpg, live_002.jpg ordered like a short video stream).
# Name them consistently and they'll play through in order.
import glob
extra_frames = sorted(glob.glob(str(REPO / 'references' / 'live_*.jpg')))
if not extra_frames:
    print('no additional frames — drop references/live_001.jpg, live_002.jpg, \u2026 for a sequence')
for p in extra_frames:
    frame = cv2.imread(p)
    if frame is None:
        print(f'{p}: could not read')
        continue
    process_frame(frame, source_label=Path(p).name)

## 5. Verify / report — did we reach the target?

**Production analog:** When the controller emits STOP and the next frame's OWLv2 detection drops to zero (the bottle is now under the scoop, not visible to the camera), the FSM transitions to VERIFYING → REPORTING (success), POSTs the result to the relay. The reporter's submission is marked complete in the database.

Here, the trajectory log shows whether the brain converged to a stable STOP.

In [ ]:
if not history:
    print('no frames processed yet')
else:
    print(f'{"#":>3}  {"action":<14}  {"dets":>4}  {"step ms":>8}  source')
    print('-' * 60)
    for i, row in enumerate(history, 1):
        act, n, ms, src, *_ = row
        print(f'{i:>3}  {act.name:<14}  {n:>4}  {ms:>8.0f}  {src}')

    last5 = [r[0].name for r in history[-5:]]
    print()
    if len(last5) == 5 and len(set(last5)) > 3:
        print('⚠️  controller is oscillating — last 5 actions are noisy. ALIGN_TOLERANCE may need tuning.')
    elif last5 == ['STOP'] * 5:
        print('✅ stable STOP — mission would now transition to VERIFYING → REPORTING (success).')
    elif Action.STOP.name in last5:
        print('🟡 STOP appeared but not yet stable — the next frame should confirm.')
    else:
        actions_seen = set(a.name for a, _, _, _ in history)
        print(f'mission in progress — actions seen: {actions_seen}')

## 6. New mission (reset)

**Production analog:** After REPORTING, the FSM returns to IDLE and polls the relay for the next pending report. Each new report = new REF/CTX = fresh controller state.

Run this cell to clear history + reset controller state, then go back to Cell 2 (upload new REF/CTX) for the next "mission."

In [ ]:
history.clear()
controller._search_remaining = 0
controller._search_direction = None
# Drop the override so the next mission re-prompts.
for var in ('REF_OVERRIDE', 'CTX_OVERRIDE'):
    globals().pop(var, None)
print('mission reset. Go to Cell 2 to upload a new report.')

## What this proves about the production system

If you run a full mission (upload REF/CTX → upload a sequence of live frames → see Actions converge to STOP), you've validated:
- The brain accepts arbitrary reporter photos and binds them as the mission target (OWLv2 query embedding cached, controller wired to both refs).
- The same code path the real robot uses (`detect → controller.step → pwm_for`) runs end-to-end on real images.
- The controller's stateful logic (search bursts, STOP detection) works across a stream of frames.
- Qwen3-VL produces usable directional guidance from the three-image prompt.

What's left for production isn't more brain code — it's wiring the brain to the Pi (already implemented in `brain/io/pi_bridge.py`), the iPhone GPS listener (already implemented in `brain/io/iphone_listener.py`), the relay backend (Node, not yet built), and the phone-side nav loop (your teammate's `robot-console/`).

## Debug report

Run the cell below at the end of a session. Everything printed is a self-contained dump (commit, GPU, models loaded, references, trajectory). Copy the output and paste it back to Claude for debugging.

In [ ]:
# --- DEBUG REPORT ---
# Run at the end of a session. Copy EVERYTHING printed and paste it back to
# Claude on the Mac. Self-contained: env, GPU, models, tuning constants,
# reference metadata (with hashes so drift is detectable), controller state,
# and per-frame detection bboxes.

import subprocess, sys, platform, hashlib, importlib, datetime, os
from pathlib import Path as _P

def _pkg_ver(name):
    try:
        m = importlib.import_module(name)
        return getattr(m, '__version__', '(no __version__)')
    except Exception as e:
        return f'(not importable: {type(e).__name__})'

def _git():
    try:
        return subprocess.check_output(
            ['git', 'log', '-1', '--pretty=format:%h %s (%ar, %an)'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return '(not a git repo)'

def _file_info(p):
    p = _P(p)
    if not p.exists():
        return f'MISSING at `{p}`'
    size = p.stat().st_size
    md5 = hashlib.md5(p.read_bytes()).hexdigest()[:10]
    try:
        import cv2
        img = cv2.imread(str(p))
        dims = f'{img.shape[1]}x{img.shape[0]}' if img is not None else 'unreadable'
    except Exception:
        dims = '?'
    return f'`{p.name}` {dims}px, {size} bytes, md5={md5}'

L = []
L.append(f'# Session report ({datetime.datetime.now().isoformat(timespec="seconds")})')
L.append('')
L.append('## Version')
L.append(f'- iteration: `{ITERATION}`' if "ITERATION" in dir() else '- iteration: UNKNOWN (banner cell not run?)')
L.append(f'- commit:    `{_git()}`')
L.append(f'- cwd:       `{os.getcwd()}`')

L.append('')
L.append('## Environment')
L.append(f'- python:        {sys.version.split()[0]}  ({platform.system()} {platform.release()})')
for pkg in ('torch', 'torchvision', 'transformers', 'accelerate', 'bitsandbytes', 'PIL', 'cv2', 'numpy', 'matplotlib'):
    L.append(f'- {pkg:<14} {_pkg_ver(pkg)}')

L.append('')
L.append('## GPU')
try:
    import torch
    L.append(f'- cuda available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        L.append(f'- device:       {torch.cuda.get_device_name(0)}')
        L.append(f'- compute cap:  {p.major}.{p.minor}')
        L.append(f'- vram total:   {p.total_memory / 1e9:.2f} GB')
        L.append(f'- vram alloc:   {torch.cuda.memory_allocated() / 1e9:.2f} GB')
        L.append(f'- vram reserved:{torch.cuda.memory_reserved() / 1e9:.2f} GB')
except Exception as e:
    L.append(f'- torch error: {e}')

L.append('')
L.append('## Models loaded')
try:
    from brain.perception.target_finder import TARGET_MIN_SIM, NMS_THRESHOLD, DEFAULT_MODEL as OWL_DEFAULT
    L.append(f'- OWLv2 class:        TargetFinder')
    L.append(f'- OWLv2 default mdl:  {OWL_DEFAULT}')
    L.append(f'- OWLv2 nms thresh:   {NMS_THRESHOLD}')
    L.append(f'- OWLv2 default sim:  {TARGET_MIN_SIM}')
    try:
        L.append(f'- OWLv2 device:       {finder.device}')
        L.append(f'- OWLv2 min_sim:      {finder.min_sim}  (effective this run)')
        L.append(f'- OWLv2 ref loaded:   {finder._query is not None}')
        if finder._query is not None:
            L.append(f'- OWLv2 ref size:     {finder._query.size}')
    except NameError:
        L.append('- OWLv2 instance: NOT loaded in this session')
except Exception as e:
    L.append(f'- OWLv2 import error: {e}')

try:
    from brain.perception.vlm_scout import DEFAULT_MODEL as VLM_DEFAULT
    L.append(f'- VLM default mdl:    {VLM_DEFAULT}')
    try:
        L.append(f'- VLM max_new_tok:    {scout.max_new_tokens}')
        mdev = next(scout.model.parameters()).device
        L.append(f'- VLM device:         {mdev}')
        L.append(f'- VLM dtype:          {next(scout.model.parameters()).dtype}')
    except NameError:
        L.append('- VLM instance: NOT loaded in this session')
except Exception as e:
    L.append(f'- VLM import error: {e}')

L.append('')
L.append('## Controller (tuning constants from brain/control/loop.py)')
try:
    from brain.control import loop as _loop
    L.append(f'- STOP_AREA_FRAC:    {_loop.STOP_AREA_FRAC}')
    L.append(f'- ALIGN_TOLERANCE:   {_loop.ALIGN_TOLERANCE}')
    L.append(f'- SEARCH_FRAMES:     {_loop.SEARCH_FRAMES}')
    try:
        L.append(f'- current _search_remaining:  {controller._search_remaining}')
        L.append(f'- current _search_direction:  {controller._search_direction!r}')
        L.append(f'- reference_photo path:       `{controller.reference_photo}`')
        L.append(f'- reporter_photo path:        `{controller.reporter_photo}`')
    except NameError:
        L.append('- controller: NOT instantiated')
except Exception as e:
    L.append(f'- controller import error: {e}')

L.append('')
L.append('## PWM mapping')
try:
    from brain.control.action_to_pwm import ACTION_TO_PWM
    for a, pwm in ACTION_TO_PWM.items():
        L.append(f'- {a.name:<14} {pwm}')
except Exception as e:
    L.append(f'- import error: {e}')

L.append('')
L.append('## References (with hashes so Claude can detect fixture drift)')
try:
    L.append(f'- REF:  {_file_info(REF)}')
    L.append(f'- CTX:  {_file_info(CTX)}')
    if 'live_path' in dir():
        L.append(f'- LIVE: {_file_info(live_path)}')
except NameError:
    L.append('- references not bound yet')

L.append('')
L.append('## Trajectory')
try:
    h = history
    if not h:
        L.append('- no frames processed yet')
    else:
        L.append(f'- {len(h)} frames processed')
        L.append('')
        L.append('| # | action | dets | ms | top detections (conf @ bbox xyxy) | source |')
        L.append('|---|---|---|---|---|---|')
        for i, row in enumerate(h, 1):
            act, n, ms, *rest = row
            src = rest[0] if rest else ''
            tops = rest[1] if len(rest) > 1 else []
            if tops:
                top_strs = [f'{c:.2f}@{xyxy}' for c, xyxy in tops[:3]]
                top_col = ' / '.join(top_strs)
            else:
                top_col = '—'
            L.append(f'| {i} | {act.name} | {n} | {ms:.0f} | {top_col} | {src} |')
        acts = [r[0].name for r in h]
        L.append('')
        L.append(f'- unique actions seen: {sorted(set(acts))}')
        L.append(f'- last 5: {acts[-5:]}')
except NameError:
    L.append('- no history variable in scope')

print('\n'.join(L))
print()
print('--- copy everything above this line and paste it back to me ---')